In [1]:
import os
import re
import numpy as np
import pandas as pd
from datetime import datetime
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

IN_DIR    = r"../../data/Riot_API/feature/datamart"
OUT_DIR   = r"../../data/Riot_API/feature/datamart/early"
PATH_EARLY = os.path.join(IN_DIR, "dm_match_team_early.csv")

print("IN :", PATH_EARLY)
print("OUT:", OUT_DIR)

# =========================
# Run config (재현성 고정)
# =========================
SEED = 42
TEST_SIZE = 0.2
SPLIT_LABEL = "matchid_random_80_20"
VARIANT = "early10_all_features"   # early는 meta 제외하고 전부 사용

# 모델 파라미터
N_ESTIMATORS = 350
LEARNING_RATE = 0.03

MAX_DEPTH = 3
NUM_LEAVES = 31

SUBSAMPLE = 0.8
COLSAMPLE = 0.8

REG_LAMBDA = 1.5
REG_ALPHA = 0.2

MIN_CHILD_SAMPLES = 200
MIN_CHILD_WEIGHT = 50

# 대시보드에서 사용할 상위 피처 개수
TOP_K_LOCAL = 15
TOPN_GLOBAL = 12

IN : ../../data/Riot_API/feature/datamart\dm_match_team_early.csv
OUT: ../../data/Riot_API/feature/datamart/early


In [2]:
df = pd.read_csv(PATH_EARLY)
print("early dm shape:", df.shape)
print("columns:", df.columns.tolist())
display(df.head(3))

# required keys
req = ["match_id", "team_id", "win_team"]
missing = [c for c in req if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in early DM: {missing}")

# (match_id, team_id) unique
dup = df.duplicated(["match_id","team_id"]).sum()
print("Duplicates (match_id, team_id):", int(dup))
assert dup == 0

# 2row check
rows_per_match = df.groupby("match_id").size()
bad = rows_per_match[rows_per_match != 2]
print("Non-2row matches:", len(bad))
assert len(bad) == 0

# team_id values
bad_team = df.loc[~df["team_id"].isin([100,200]), "team_id"].unique()
print("team_id unique:", df["team_id"].unique())
assert len(bad_team) == 0

# win_team per match should sum to 1
wt = df["win_team"]
if wt.dtype == "object":
    wt_norm = wt.astype(str).str.lower().map({"true":1,"false":0,"1":1,"0":0})
else:
    wt_norm = wt.astype(int)

win_sum = wt_norm.groupby(df["match_id"]).sum()
bad_win = win_sum[win_sum != 1]
print("Bad matches (win_team sum != 1):", len(bad_win))
assert len(bad_win) == 0

early dm shape: (108412, 24)
columns: ['match_id', 'platform', 'team_id', 'queue_id', 'game_duration', 'game_mode', 'game_version', 'game_creation', 'win_team', 'gold_diff_10', 'xp_diff_10', 'cs_diff_10', 'dmg_to_champ_diff_10', 'dmg_taken_diff_10', 'level_diff_10', 'enemy_cc_time_diff_10', 'evt_champion_kill_10_diff_10', 'evt_elite_monster_kill_10_diff_10', 'evt_turret_plate_destroyed_10_diff_10', 'evt_ward_kill_10_diff_10', 'evt_ward_placed_10_diff_10', 'first_blood_diff', 'elite_dragon_10_diff_10', 'elite_horde_10_diff_10']


,match_id,platform,team_id,queue_id,game_duration,game_mode,game_version,game_creation,win_team,gold_diff_10,xp_diff_10,cs_diff_10,dmg_to_champ_diff_10,dmg_taken_diff_10,level_diff_10,enemy_cc_time_diff_10,evt_champion_kill_10_diff_10,evt_elite_monster_kill_10_diff_10,evt_turret_plate_destroyed_10_diff_10,evt_ward_kill_10_diff_10,evt_ward_placed_10_diff_10,first_blood_diff,elite_dragon_10_diff_10,elite_horde_10_diff_10
0,BR1_3050794586,br1,100,440,1597,CLASSIC,15.1.647.8644,1736438364688,False,3078.0,2296.0,20.0,4844.0,-2144.0,3.0,63.140,7,-4,0,1,8,1,0.0,0.0
1,BR1_3050794586,br1,200,440,1597,CLASSIC,15.1.647.8644,1736438364688,True,-3078.0,-2296.0,-20.0,-4844.0,2144.0,-3.0,-63.140,-7,4,0,-1,-8,-1,0.0,0.0
2,BR1_3050880271,br1,100,420,2073,CLASSIC,15.1.647.8644,1736450995040,False,70.0,11.0,13.0,-3147.0,6704.0,0.0,-329.153,-6,2,0,-1,-319,-1,-1.0,3.0


Duplicates (match_id, team_id): 0
Non-2row matches: 0
team_id unique: [100 200]
Bad matches (win_team sum != 1): 0


In [3]:
# target
y = df["win_team"].astype(int).values

# meta columns (filter/join용, 모델 입력에서는 제외)
meta_cols = [
    "match_id", "platform", "team_id",
    "queue_id", "game_mode", "game_version",
    "game_creation", "game_duration",  # early에서 누수 가능성 -> meta로 고정
    "win_team",
]
meta_cols = [c for c in meta_cols if c in df.columns]

# feature columns = all - meta
feature_cols = [c for c in df.columns if c not in meta_cols]

print("meta_cols:", meta_cols)
print("feature_cols:", len(feature_cols), feature_cols)

# sanity: numeric only
non_numeric = [c for c in feature_cols if df[c].dtype == "object"]
print("non-numeric feature cols:", non_numeric)
assert len(non_numeric) == 0, "Early feature에 object dtype이 있습니다. 숫자로 변환/정리 필요."

# optional leak scan
leak_pat = re.compile(r"(surrender|end_of_game|game_end|ended|remake|afk)", re.IGNORECASE)
leak_cols = [c for c in feature_cols if leak_pat.search(c)]
print("leak-like cols:", leak_cols)
assert len(leak_cols) == 0, "Early feature에 누수 패턴이 감지되었습니다."

meta_cols: ['match_id', 'platform', 'team_id', 'queue_id', 'game_mode', 'game_version', 'game_creation', 'game_duration', 'win_team']
feature_cols: 15 ['gold_diff_10', 'xp_diff_10', 'cs_diff_10', 'dmg_to_champ_diff_10', 'dmg_taken_diff_10', 'level_diff_10', 'enemy_cc_time_diff_10', 'evt_champion_kill_10_diff_10', 'evt_elite_monster_kill_10_diff_10', 'evt_turret_plate_destroyed_10_diff_10', 'evt_ward_kill_10_diff_10', 'evt_ward_placed_10_diff_10', 'first_blood_diff', 'elite_dragon_10_diff_10', 'elite_horde_10_diff_10']
non-numeric feature cols: []
leak-like cols: []


In [4]:
def early_feature_group(c: str) -> str:
    lc = c.lower()
    # ECON
    if any(k in lc for k in ["gold_10", "xp_10", "cs_10", "level_10"]):
        return "EARLY_ECON"
    # COMBAT
    if any(k in lc for k in ["dmg_to_champ", "dmg_taken", "enemy_cc_time", "evt_champion_kill"]):
        return "EARLY_COMBAT"
    # VISION
    if "ward" in lc:
        return "EARLY_VISION"
    # OBJECTIVE (early 전환 이벤트 포함)
    if any(k in lc for k in ["elite", "evt_elite_monster_kill", "evt_building_kill", "turret_plate_destroyed"]):
        return "EARLY_OBJECTIVE"
    return "EARLY_OTHER"

# quick check
grp_counts = {}
for c in feature_cols:
    g = early_feature_group(c)
    grp_counts[g] = grp_counts.get(g, 0) + 1
print("early group counts:", grp_counts)

early group counts: {'EARLY_OTHER': 5, 'EARLY_COMBAT': 4, 'EARLY_OBJECTIVE': 4, 'EARLY_VISION': 2}


In [5]:
X = df[feature_cols].copy()

match_ids = df["match_id"].unique()
train_m, test_m = train_test_split(match_ids, test_size=TEST_SIZE, random_state=SEED, shuffle=True)

train_mask = df["match_id"].isin(train_m)
test_mask  = df["match_id"].isin(test_m)

X_train = X.loc[train_mask].reset_index(drop=True)
y_train = y[train_mask]
X_test  = X.loc[test_mask].reset_index(drop=True)
y_test  = y[test_mask]

idx_train = df.index[train_mask].values
idx_test  = df.index[test_mask].values

print("train rows:", X_train.shape, "test rows:", X_test.shape)
print("train matches:", len(train_m), "test matches:", len(test_m))
print("train win rate:", y_train.mean(), "test win rate:", y_test.mean())

overlap = len(set(df.loc[idx_train, "match_id"]) & set(df.loc[idx_test, "match_id"]))
print("overlap matches:", overlap)
assert overlap == 0, "Split leakage: same match_id appears in both train and test!"

train rows: (86728, 15) test rows: (21684, 15)
train matches: 43364 test matches: 10842
train win rate: 0.5 test win rate: 0.5
overlap matches: 0


In [6]:
def train_lgbm(X_train, y_train):
    model = lgb.LGBMClassifier(
        n_estimators=N_ESTIMATORS,
        learning_rate=LEARNING_RATE,
        num_leaves=NUM_LEAVES,
        max_depth=MAX_DEPTH,
        min_child_samples=MIN_CHILD_SAMPLES,
        subsample=SUBSAMPLE,
        colsample_bytree=COLSAMPLE,
        reg_lambda=REG_LAMBDA,
        reg_alpha=REG_ALPHA,
        random_state=SEED,
        n_jobs=-1,
        boosting_type="gbdt",
    )
    model.fit(X_train, y_train)
    return model

def train_xgb(X_train, y_train):
    model = xgb.XGBClassifier(
        n_estimators=N_ESTIMATORS,
        learning_rate=LEARNING_RATE,
        max_depth=MAX_DEPTH,
        min_child_weight=MIN_CHILD_WEIGHT,
        subsample=SUBSAMPLE,
        colsample_bytree=COLSAMPLE,
        reg_lambda=REG_LAMBDA,
        reg_alpha=REG_ALPHA,
        random_state=SEED,
        n_jobs=-1,
        eval_metric="logloss",
        tree_method="hist",
    )
    model.fit(X_train, y_train)
    return model

def eval_model(model, X_train, y_train, X_test, y_test):
    p_train = model.predict_proba(X_train)[:, 1]
    p_test  = model.predict_proba(X_test)[:, 1]
    return {
        "auc_train": roc_auc_score(y_train, p_train),
        "auc_test": roc_auc_score(y_test, p_test),
        "logloss_train": log_loss(y_train, p_train),
        "logloss_test": log_loss(y_test, p_test),
        "p_train": p_train,
        "p_test": p_test,
    }

In [7]:
import shap

def make_explainer(model):
    return shap.TreeExplainer(model, model_output="raw")  # raw=logit

def unwrap_shap(shap_values):
    if isinstance(shap_values, list):
        return shap_values[1]
    return shap_values

def unwrap_base(expected_value):
    if isinstance(expected_value, (list, np.ndarray)):
        return float(expected_value[1] if len(expected_value) > 1 else expected_value[0])
    return float(expected_value)

In [8]:
def build_global_shap(model_name, base_logit, base_prob, phi, feat_names):
    mean_abs_shap = np.mean(np.abs(phi), axis=0)
    mean_signed_shap = np.mean(phi, axis=0)

    dp = sigmoid(base_logit + phi) - base_prob
    mean_abs_dp = np.mean(np.abs(dp), axis=0)
    mean_signed_dp = np.mean(dp, axis=0)

    out = pd.DataFrame({
        "model_name": model_name,
        "variant": VARIANT,
        "split": SPLIT_LABEL,
        "seed": SEED,
        "n_estimators": N_ESTIMATORS,
        "base_logit": base_logit,
        "base_prob": base_prob,
        "feature_name": feat_names,
        "early_group": [early_feature_group(f) for f in feat_names],
        "mean_abs_shap_logit": mean_abs_shap,
        "mean_signed_shap_logit": mean_signed_shap,
        "mean_abs_delta_p": mean_abs_dp,
        "mean_signed_delta_p": mean_signed_dp,
    }).sort_values("mean_abs_shap_logit", ascending=False).reset_index(drop=True)

    out["rank"] = np.arange(1, len(out) + 1)
    return out

def build_group_shap(model_name, base_logit, base_prob, phi, feat_cols_used):
    groups = ["EARLY_ECON","EARLY_COMBAT","EARLY_VISION","EARLY_OBJECTIVE","EARLY_OTHER"]

    group_to_idx = {}
    for g in groups:
        idx = [i for i, f in enumerate(feat_cols_used) if early_feature_group(f) == g]
        group_to_idx[g] = idx

    rows=[]
    for g, idxs in group_to_idx.items():
        if len(idxs) == 0:
            continue
        g_logit = phi[:, idxs].sum(axis=1)
        g_dp = sigmoid(base_logit + g_logit) - base_prob
        rows.append({
            "model_name": model_name,
            "variant": VARIANT,
            "split": SPLIT_LABEL,
            "seed": SEED,
            "n_estimators": N_ESTIMATORS,
            "base_logit": base_logit,
            "base_prob": base_prob,
            "group": g,
            "n_features_in_group": len(idxs),
            "mean_abs_shap_logit": float(np.mean(np.abs(g_logit))),
            "mean_signed_shap_logit": float(np.mean(g_logit)),
            "mean_abs_delta_p": float(np.mean(np.abs(g_dp))),
            "mean_signed_delta_p": float(np.mean(g_dp)),
        })

    out = pd.DataFrame(rows).sort_values("mean_abs_shap_logit", ascending=False).reset_index(drop=True)
    out["rank"] = np.arange(1, len(out) + 1)
    return out

def build_local_shap(model_name, base_logit, base_prob, phi, feat_names, id_df, pred_prob, top_k=15):
    rows=[]
    for i in range(phi.shape[0]):
        row_phi = phi[i, :]
        abs_phi = np.abs(row_phi)
        top_idx = np.argsort(-abs_phi)[:top_k]
        top_phi = row_phi[top_idx]
        top_dp = sigmoid(base_logit + top_phi) - base_prob

        tmp = pd.DataFrame({
            "model_name": model_name,
            "variant": VARIANT,
            "split": SPLIT_LABEL,
            "seed": SEED,
            "n_estimators": N_ESTIMATORS,
            "match_id": id_df.loc[i, "match_id"],
            "team_id": id_df.loc[i, "team_id"],
            "pred_prob_early": float(pred_prob[i]),
            "feature_name": feat_names[top_idx],
            "early_group": [early_feature_group(f) for f in feat_names[top_idx]],
            "shap_logit": top_phi,
            "abs_shap_logit": np.abs(top_phi),
            "delta_p": top_dp,
            "abs_delta_p": np.abs(top_dp),
        }).sort_values("abs_shap_logit", ascending=False).reset_index(drop=True)

        tmp["rank_in_row"] = np.arange(1, len(tmp) + 1)
        rows.append(tmp)

    return pd.concat(rows, ignore_index=True)

In [9]:
feat_names = np.array(feature_cols)

# test ids (join key)
id_df_test = df.loc[idx_test, ["match_id","team_id"]].reset_index(drop=True)

global_all=[]
group_all=[]
local_all=[]
metrics_rows=[]

for model_name in ["lightgbm", "xgboost"]:
    try:
        if model_name == "lightgbm":
            model = train_lgbm(X_train, y_train)
        else:
            model = train_xgb(X_train, y_train)

        met = eval_model(model, X_train, y_train, X_test, y_test)

        print(f"\n[{model_name}] AUC train={met['auc_train']:.4f}, test={met['auc_test']:.4f} | "
              f"Logloss train={met['logloss_train']:.4f}, test={met['logloss_test']:.4f}")

        explainer = make_explainer(model)
        shap_test = unwrap_shap(explainer.shap_values(X_test))
        base_logit = unwrap_base(explainer.expected_value)
        base_prob = float(sigmoid(base_logit))

        gdf = build_global_shap(model_name, base_logit, base_prob, shap_test, feat_names)
        grdf = build_group_shap(model_name, base_logit, base_prob, shap_test, feature_cols)
        ldf = build_local_shap(model_name, base_logit, base_prob, shap_test, feat_names, id_df_test, met["p_test"], top_k=TOP_K_LOCAL)

        global_all.append(gdf)
        group_all.append(grdf)
        local_all.append(ldf)

        metrics_rows.append({
            "model_name": model_name,
            "variant": VARIANT,
            "split": SPLIT_LABEL,
            "seed": SEED,
            "n_estimators": N_ESTIMATORS,
            "learning_rate": LEARNING_RATE,
            "n_features": int(X_train.shape[1]),
            "n_train_rows": int(X_train.shape[0]),
            "n_test_rows": int(X_test.shape[0]),
            "n_train_matches": int(len(train_m)),
            "n_test_matches": int(len(test_m)),
            "auc_train": met["auc_train"],
            "auc_test": met["auc_test"],
            "logloss_train": met["logloss_train"],
            "logloss_test": met["logloss_test"],
            "base_logit": base_logit,
            "base_prob": base_prob,
        })

    except Exception as e:
        print(f"[{model_name}] ERROR:", repr(e))

global_df = pd.concat(global_all, ignore_index=True)
group_df  = pd.concat(group_all, ignore_index=True)
local_df  = pd.concat(local_all, ignore_index=True)
metrics_df = pd.DataFrame(metrics_rows)

# ---- Save core outputs ----
OUT_GLOBAL  = os.path.join(OUT_DIR, "dm_match_team_shap_global_early.csv")
OUT_GROUP   = os.path.join(OUT_DIR, "dm_match_team_shap_group_early.csv")
OUT_METRICS = os.path.join(OUT_DIR, "dm_model_metrics_early.csv")

global_df.to_csv(OUT_GLOBAL, index=False)
group_df.to_csv(OUT_GROUP, index=False)
metrics_df.to_csv(OUT_METRICS, index=False)

print("\nSaved core:")
print(" -", OUT_GLOBAL, global_df.shape)
print(" -", OUT_GROUP, group_df.shape)
print(" -", OUT_METRICS, metrics_df.shape)

display(metrics_df)

[LightGBM] [Info] Number of positive: 43364, number of negative: 43364
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001082 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1913
[LightGBM] [Info] Number of data points in the train set: 86728, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with pos

c:\Users\andor\AppData\Local\Programs\Python\Python311\Lib\site-packages\shap\explainers\_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(



[xgboost] AUC train=0.7457, test=0.7365 | Logloss train=0.5941, test=0.6020

Saved core:
 - ../../data/Riot_API/feature/datamart/early\dm_match_team_shap_global_early.csv (30, 14)
 - ../../data/Riot_API/feature/datamart/early\dm_match_team_shap_group_early.csv (8, 14)
 - ../../data/Riot_API/feature/datamart/early\dm_model_metrics_early.csv (2, 17)


,model_name,variant,split,seed,n_estimators,learning_rate,n_features,n_train_rows,n_test_rows,n_train_matches,n_test_matches,auc_train,auc_test,logloss_train,logloss_test,base_logit,base_prob
0,lightgbm,early10_all_features,matchid_random_80_20,42,350,0.03,15,86728,21684,43364,10842,0.745629,0.736094,0.59405,0.602321,0.000015,0.500004
1,xgboost,early10_all_features,matchid_random_80_20,42,350,0.03,15,86728,21684,43364,10842,0.745707,0.736488,0.59407,0.602013,0.000301,0.500075


In [10]:
# join 가능한 meta subset (early dm에 있는 것만)
meta_for_join = [c for c in ["match_id","team_id","platform","queue_id","game_version","game_creation","game_mode","win_team"] if c in df.columns]
meta_df = df.loc[idx_test, meta_for_join].reset_index(drop=True)

# local은 match_id/team_id 중복 없이 하나당 TOP_K_LOCAL rows라 merge시 key 기준으로 확장됨
local_with_meta = local_df.merge(meta_df, on=["match_id","team_id"], how="left")

OUT_LOCAL_META = os.path.join(OUT_DIR, "dm_match_team_shap_local_early_with_meta.csv")
local_with_meta.to_csv(OUT_LOCAL_META, index=False)

print("Saved local with meta:")
print(" -", OUT_LOCAL_META, local_with_meta.shape)

Saved local with meta:
 - ../../data/Riot_API/feature/datamart/early\dm_match_team_shap_local_early_with_meta.csv (650520, 21)


In [11]:
# local extreme rate
for model_name in local_df["model_name"].unique():
    key = local_df[local_df["model_name"]==model_name].groupby(["match_id","team_id"], as_index=False)["pred_prob_early"].first()
    per_match = key.groupby("match_id")["pred_prob_early"].agg(["count","min","max"])
    extreme_rate = ((per_match["max"]>0.995) & (per_match["min"]<0.005)).mean()
    print(f"{model_name}: test_matches={len(per_match)}, extreme_rate={extreme_rate:.1%}")

# group importance order 확인
for model_name in group_df["model_name"].unique():
    sub = group_df[group_df["model_name"]==model_name].sort_values("rank")
    print("\n", model_name, "group ranks:")
    print(sub[["rank","group","mean_abs_delta_p","n_features_in_group"]].to_string(index=False))

lightgbm: test_matches=10842, extreme_rate=0.0%
xgboost: test_matches=10842, extreme_rate=0.0%

 lightgbm group ranks:
 rank           group  mean_abs_delta_p  n_features_in_group
    1     EARLY_OTHER          0.117743                    5
    2 EARLY_OBJECTIVE          0.063524                    4
    3    EARLY_COMBAT          0.051182                    4
    4    EARLY_VISION          0.003945                    2

 xgboost group ranks:
 rank           group  mean_abs_delta_p  n_features_in_group
    1     EARLY_OTHER          0.116811                    5
    2 EARLY_OBJECTIVE          0.063365                    4
    3    EARLY_COMBAT          0.052908                    4
    4    EARLY_VISION          0.004760                    2
